# QuantLLMBot Phase 4 — Qwen2.5-14B QLoRA (Colab)

Full chain in Colab: preprocess → finetune → evaluate → **export ready Ollama model**.

**Runtime:** GPU — **A100 40GB** required for the 14B merge in step 9 (L4 24GB can train but not merge; T4: not enough).

**You upload:** `QuantLLMBot_training.zip` (46 KB).
**You download at the end:** `qwen-trading-v003-14b-q4_k_m.gguf` (~9 GB) + `Modelfile` from your Google Drive → paste into Ollama on the PC.

Disk use in Colab: ~70 GB temp (merged model + GGUF) — fits the A100 runtime disk. Only ~9 GB goes to your Drive.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload QuantLLMBot_training.zip and extract
from google.colab import files
up = files.upload()  # pick QuantLLMBot_training.zip from your PC

!unzip -o -q /content/QuantLLMBot_training.zip -d /content/QuantLLMBot

import os
os.environ['QUANTLLM_ROOT'] = '/content/QuantLLMBot'
%cd /content/QuantLLMBot/scripts

In [ ]:
# 3. Install dependencies (~2 min)
!pip install -q -r requirements.txt

In [ ]:
# 4. Pre-flight checks — everything must be green before cell 6
!python pre_training_checklist.py

In [ ]:
# 5. Preprocess — expect "379 instruction-response pairs" (v7: 379 train / 10 holdout)
!python 01_preprocess.py

In [ ]:
# 6. Fine-tune Qwen2.5-14B with QLoRA
# First run downloads ~15 GB of model weights, then trains (~30-60 min on A100)
!python 02_finetune.py

In [ ]:
# 7. Evaluate on the 10 held-out examples
!python 03_evaluate.py
!cat /content/QuantLLMBot/model_training/outputs/evaluation_results.json

In [ ]:
# 8. (Backup) Download just the LoRA adapter — small, keep it even if you export the GGUF
!cd /content/QuantLLMBot/model_training/outputs && zip -qr qwen14b_lora_weights.zip lora_weights
from google.colab import files
files.download('/content/QuantLLMBot/model_training/outputs/qwen14b_lora_weights.zip')

---
## Export ready Ollama model (GGUF Q4_K_M)

Merges the LoRA into the base 14B, converts to GGUF, quantizes to Q4_K_M, writes a Modelfile, and copies the final ~9 GB package to your Google Drive.

In [ ]:
# 9. Mount Google Drive (final ~9 GB package is saved there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 10. Free VRAM from training, install GGUF deps, fix peft/torchao clash
import gc, torch
gc.collect(); torch.cuda.empty_cache()
# Colab ships torchao 0.10; peft merge needs >=0.16 or torchao removed
!pip install -q -U 'torchao>=0.16.0' gguf sentencepiece
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# 11. Merge LoRA -> GGUF f16 -> quantize Q4_K_M -> Modelfile (~20-30 min)
# Needs A100 40GB for fp16 merge. Script auto-fixes Colab torchao/peft clash.
import gc, torch
gc.collect(); torch.cuda.empty_cache()
!pip install -q -U 'torchao>=0.16.0' gguf sentencepiece
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

!python /content/QuantLLMBot/model_training/export_lora_to_ollama_colab.py \
    --base-model Qwen/Qwen2.5-14B-Instruct \
    --adapter-dir /content/QuantLLMBot/model_training/outputs/lora_weights \
    --work-dir /content/ollama_export \
    --outfile-prefix qwen-trading-v003-14b \
    --quant Q4_K_M

In [ ]:
# 12. Copy ready package to Google Drive (~9 GB, takes a few minutes)
!mkdir -p /content/drive/MyDrive/QuantLLMBot/ollama_v003
!cp /content/ollama_export/ollama_model/qwen-trading-v003-14b-q4_k_m.gguf \
    /content/ollama_export/ollama_model/Modelfile \
    /content/drive/MyDrive/QuantLLMBot/ollama_v003/
!ls -lh /content/drive/MyDrive/QuantLLMBot/ollama_v003/
print('\nDONE — on your PC:')
print('1. Download the ollama_v003 folder from Google Drive')
print('2. In that folder run:  ollama create qwen-trading-v003 -f Modelfile')
print('3. Test:                ollama run qwen-trading-v003')